In [ ]:
import copy
import math
from typing import List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, Dataset
import matplotlib.pyplot as plt
import flwr_datasets
from datasets import concatenate_datasets


In [ ]:
num_helpers = 3
num_clients = 4

In [ ]:
class HFWrapper(Dataset):
    def __init__(
        self, hf_dataset, image_key="image", label_key="character",
        # transform=transforms.ToTensor()
    ):
        self.hf = hf_dataset
        self.image_key = image_key
        self.label_key = label_key
        # self.transform = transforms.ToTensor()
        # if transform:
        #     self.transform = transform
        self.transform = transforms.ToTensor()
        # print("The size of the dataset is: ", len(hf_dataset))
        # self.transform = transform if transform is not None else transforms.ToTensor()

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, idx):
        item = self.hf[idx]
        img = item[self.image_key]
        label = item[self.label_key]
        img = self.transform(img)
        # print(f"debug: {type(label)}")
        # label = int(label)
        return img, label



    # return DataLoader(
    #     wrapped, batch_size=batch_size, shuffle=shuffle, pin_memory=False, num_workers=0
    # )



def make_loader(hf_dataset, batch_size=32, shuffle=False):
    wrapped = HFWrapper(hf_dataset)
    return DataLoader(
        wrapped,
        batch_size=batch_size,
        shuffle=shuffle,
    )

In [ ]:
class FEMNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 7, padding=3)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc = nn.Linear(64 * 7 * 7, 62)   # 62 classes in FEMNIST

    def forward(self, x):
        x = x.reshape(-1, 1, 28, 28)
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = x.flatten(1)
        return self.fc(x)

In [ ]:
class Helper(dict):
    def __init__(self,idx):
        self.hid = idx


helper_nodes = []
for i in range(num_helpers):
    helper_nodes.append(Helper(i))
# helper_nodes = [Helper(1),Helper(2),Helper(),Helper()]

In [ ]:
import random
# def client_to_helper(cid,item):
#     eve = random.randint(0,9)
#     items = 2*[item]
#     if eve<2:
#         items[eve]=None
#     helper_nodes[0][cid]=items[0]
#     helper_nodes[1][cid]=items[1]
#     # helper_nodes[cid][cid]=(cid,items[0])
#     # helper_nodes[cid+1][cid]=(cid,items[1])

import torch

def client_to_helper(cid,item):
    state_dict = item[0]
    num_samples = item[1]
    first_piece = {}
    second_piece = {}
    parity_piece = {}
    # for key in state_dict.keys():
    #     print(key)
    for name, param in state_dict.items():
        # Only split tensors where the first dimension is divisible by 2 and > 1 (typically weights)
        if param.dim() > 0 and param.size(0) % 2 == 0 and param.size(0) > 1:
            half = param.size(0) // 2

            first_half = param[:half].clone()
            second_half = param[half:].clone()
            parity = first_half + second_half

            first_piece[name] = first_half
            second_piece[name] = second_half
            parity_piece[name] = parity
        # else:
        #     first_piece[name] = param.clone()
        #     second_piece[name] = param.clone()
        #     parity_piece[name] = param.clone()
    codes = [first_piece,second_piece,parity_piece]
    eve = random.randint(0,9)
    helper_nodes[0][cid]=(codes[0],num_samples)
    helper_nodes[1][cid]=(codes[1],num_samples)
    helper_nodes[2][cid]=(codes[2],num_samples)
    if eve<3:
        helper_nodes[eve][cid]=None
    return first_piece, second_piece, parity_piece

def reconstruct_state(code):
    num_samp = 0
    # fp,sp,tp = dict()
    fp = dict()
    sp = dict()
    tp = dict()
    if code[0] is not None:
        num_samp = code[0][1]
    else:
        num_samp = code[1][1]
    if code[0] is None:
        sp:dict
        sp,tp = code[1][0],code[2][0]
        fp = dict()
        for key in sp.keys():
            fp[key]=tp[key]-sp[key]
    elif code[1] is None:
        fp,tp = code[0][0],code[2][0]
        sp = dict()
        for key in fp.keys():
            sp[key]=tp[key]-fp[key]
    else:
        fp,sp = code[0][0],code[1][0]
    model_state = dict()
    for key in fp.keys():
        model_state[key]=torch.cat((fp[key],sp[key]),dim=0)
    # print(len(model_state))
    # for key in model_state.keys():
    #     print(key)
    return (model_state,num_samp)
        
    # fp,sp,tp = code[0][0],code[0][1],code[0][2] # three pieces of codes
    
    
def helper_to_client():
    updates = []
    # updates = num_clients*[None]
    # for h in helper_nodes:
    #     # print(type(h))
    #     for entry in h.items():
    #         # print(type(entry[1]))
    #         if entry[1] is not None :
    #             updates[entry[0]]=entry[1]
    client_wts = num_clients*[None]
    for i in range(num_clients):
        client_wts[i]=[helper_nodes[0][i],helper_nodes[1][i],helper_nodes[2][i]]
    for code in client_wts:
        updates.append(reconstruct_state(code))
    return updates

In [ ]:
class Client:
    def __init__(self,cid, dataset, device="cpu"):
        self.cid = cid
        self.dataset = dataset
        self.device = device

    def train_local(self, global_model, epochs=1, batch_size=32, lr=0.01):
        model = copy.deepcopy(global_model).to(self.device)
        loader = make_loader(self.dataset, batch_size=batch_size, shuffle=True)
        optimizer = optim.SGD(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        model.train()
        for _ in range(epochs):
            for X, y in loader:
                X, y = X.to(self.device), y.to(self.device)
                optimizer.zero_grad()
                loss = criterion(model(X), y)
                loss.backward()
                optimizer.step()

        # return model weights + number of samples trained
        # print(model.state_dict().items())
        # for name, param in model.state_dict().items():
        #     print(f"{name}: {param.shape}")
        client_to_helper(self.cid,(model.state_dict(), len(self.dataset)))
        return model.state_dict(), len(self.dataset)


In [ ]:
class Master:
    def __init__(self, global_model, clients, device="cpu"):
        self.global_model = global_model
        self.clients = clients
        self.device = device

    def aggregate(self, updates):
        total_samples = sum(num for _, num in updates)
        new_state = {}

        for key in updates[0][0].keys():
            weighted = sum(
                (client_state[key] * (num / total_samples))
                for client_state, num in updates
            )
            new_state[key] = weighted

        self.global_model.load_state_dict(new_state)

    def run_round(self, epochs=1, batch_size=32, lr=0.01):
        updates = num_clients*[None]
        for client in self.clients:
            client_state, num_samples = client.train_local(
                self.global_model, epochs=epochs, batch_size=batch_size, lr=lr
            )
            
            
            # updates.append((client_state, num_samples))
        # for h in helper_nodes:
        #     for c in h:
        updates = helper_to_client()
        self.aggregate(updates)

    def evaluate(self, test_dataset, batch_size=64):
        loader = make_loader(test_dataset, batch_size=batch_size)
        model = self.global_model.to(self.device)
        model.eval()

        total, correct, total_loss = 0, 0, 0
        criterion = nn.CrossEntropyLoss()

        with torch.no_grad():
            for X, y in loader:
                X, y = X.to(self.device), y.to(self.device)
                outputs = model(X)
                total_loss += criterion(outputs, y).item() * y.size(0)
                correct += (outputs.argmax(1) == y).sum().item()
                total += y.size(0)

        return total_loss / total, correct / total

In [ ]:
def weighted_average(updates):
    """Weighted average of model weights based on dataset sizes."""
    total_examples = sum(num for _, num in updates)
    new_state = {}

    for key in updates[0][0].keys():
        new_state[key] = sum((state[key] * num for state, num in updates), 
                             torch.zeros_like(updates[0][0][key]))
        new_state[key] /= total_examples

    return new_state, total_examples

In [ ]:
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner, GroupedNaturalIdPartitioner
from datasets import ClassLabel
from torch.utils.data import DataLoader

num_clients = 4
num_helper = 3


# partitioner = IidPartitioner(num_partitions=num_clients)
# partitioner = Partitioner
# partitioner = GroupedNaturalIdPartitioner(num_partitions=num_clients, id_column="writer_id")
partitioner = GroupedNaturalIdPartitioner(partition_by='writer_id',group_size=10)
dataset = FederatedDataset(
    dataset="flwrlabs/femnist", partitioners={"train": partitioner}
)

partition  = dataset.load_partition(0,'train')
partition.set_format(type='torch',columns=['character'])
print(partition.features)

client_subsets = []
test_datasubsets = []


# def transform_batch(batch):
#     batch["image"] = [to_tensor(img) for img in batch["image"]]   # PIL → Tensor
#     batch["character"] = [int(lbl) for lbl in batch["character"]] # ClassLabel → int
#     return batch

for i in range(num_clients):
    curr_part = dataset.load_partition(partition_id=i, split="train")
    # curr_part = curr_part.select(range(2000))
    # curr_part = curr_part.select(range(2)
    split_data = curr_part.train_test_split(test_size=0.2, seed=42)    
    assert len(split_data['test'])>0 and len(split_data['train'])>0
    client_subsets.append(split_data["train"])
    test_datasubsets.append(split_data["test"])
clients = [Client(cid=i, dataset=client_subsets[i]) for i in range(num_clients)]


train_data = concatenate_datasets(client_subsets)
train_loader = make_loader(train_data, batch_size=256, shuffle=True)
test_data = concatenate_datasets(test_datasubsets)
test_loader = make_loader(test_data,batch_size=256,shuffle=False)





In [ ]:
cl_acc = []
def train_central(model, train_loader, test_loader, device='cpu', epochs=5):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
        
        # Evaluate
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                pred = out.argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
        acc = correct / total
        # print("Dataset size is:",len(train_loader))
        print(f"Epoch {epoch+1}: Test Accuracy: {acc*100:.2f}%")
        cl_acc.append(acc*100)



In [ ]:

global_model = FEMNISTNet()
master = Master(global_model, clients, device="cpu")

fl_acc = []
fl_loss = []
ROUNDS = 2
for r in range(1, ROUNDS + 1):
    print(f"--- Round {r} ---")
    master.run_round(epochs=2, batch_size=32, lr=0.05)
    loss, acc = master.evaluate(test_data)
    print(f"Test Loss: {loss:.4f}, Test Accuracy: {acc*100:.2f}%")
    fl_acc.append(acc*100)
    fl_loss.append(loss)

In [ ]:
print("dataset size is:",len(train_data))
central_model = FEMNISTNet()
train_central(central_model,train_loader,test_loader,device='cpu',epochs=4)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(fl_acc,label="Federate Learning Accuracy")
plt.plot(cl_acc,label="Centralised Learning Accuracy")
plt.legend()
plt.xlabel("Rounds")
plt.ylabel("Accuracy in %")
plt.show()
